# Signal Idea 5: Markov Chain on Credit-Macro Transition States

Research notebook capturing one implementable signal concept from quant literature (non-HFT, not price-only).


In [ ]:
# Setup: imports and robust paths
import sys
from pathlib import Path

import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES_ROOT = [CWD, CWD.parent]

for root in CANDIDATES_ROOT:
    if (root / "config" / "settings.py").exists():
        PROJECT_ROOT = root
        break
else:
    PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def resolve_first_existing(paths: list[Path], fallback: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    return fallback

DATA_DIR = resolve_first_existing(
    [root / "data" / "factors" for root in CANDIDATES_ROOT],
    PROJECT_ROOT / "data" / "factors",
)
DUCKDB_PATH = resolve_first_existing(
    [root / "data" / "factors" / "factors.duckdb" for root in CANDIDATES_ROOT],
    DATA_DIR / "factors.duckdb",
)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"DuckDB path  : {DUCKDB_PATH}")
print(f"Data directory exists: {DATA_DIR.exists()}")
print(f"DuckDB exists: {DUCKDB_PATH.exists()}")
if not DUCKDB_PATH.exists():
    print("⚠️ DuckDB file missing. Use parquet files directly for prototype tests.")


In [ ]:
from IPython.display import display
# Quick local data inventory for feasibility
candidates = [
    DATA_DIR / "prices.parquet",
    DATA_DIR / "factors_price.parquet",
    DATA_DIR / "factors_all.parquet",
    DATA_DIR / "macro.parquet",
    DATA_DIR / "macro_z.parquet",
]

rows = []
for p in candidates:
    rows.append({
        "file": p.name,
        "exists": p.exists(),
        "size_mb": round(p.stat().st_size / (1024**2), 2) if p.exists() else None,
    })

display_df = pd.DataFrame(rows)
display_df


## What this idea is
Discretize macro-credit conditions into finite states (e.g., {expansion, slowdown, stress}) and estimate transition probabilities with a Markov chain.

Signal uses **next-state probability** to tilt factors:
- High P(stress next month): reduce cyclical/high-beta exposure
- High P(expansion): increase pro-cyclical sleeve

This is explicitly Markov-chain based and combines macro + cross-sectional features.


## Paper lineage
- Kritzman, Li, Page, Rigobon (2012), *Principal Components as a Measure of Systemic Risk* (state framing inspiration).
- Broad regime-transition and credit-cycle Markov modeling literature in asset allocation.

Why relevant: interpretable state machine with low model complexity and strong economic priors.


## Feasibility in this project
**Feasibility: High**

- Existing `macro.parquet` / `macro_z.parquet` likely contain enough variables to define states.
- Implementation is straightforward with `pandas` + `numpy` transition matrix logic.
- Works at monthly frequency, robust to noisy daily data.


## Data requirements and availability
- **Already local**: macro parquet files and market prices.
- **Optional external**: FRED credit spreads (BAA-AAA, HY OAS proxies) if not present.

FRED data is free and easy to ingest, so missing-series risk is low.


## Minimal prototype notebook cell
```python
# pseudo-outline
# 1) Build monthly macro feature panel
# 2) Assign each month to discrete state via thresholds/clustering
# 3) Estimate transition matrix P(S_t+1 | S_t)
# 4) Trade based on one-step-ahead stress probability
```


## Recommended next experiment in this repo
1. Build a minimal feature table using currently available parquet files.
2. Run a simple walk-forward backtest using existing portfolio metrics.
3. Add one external dataset only if it materially improves explanatory power.
